#Curated Table

##Data Definitions

In [0]:
%run /Workspace/Users/greyce.costa@thoughtworks.com/ml_training_dev.olist/silver/Commons

In [0]:
CATALOG_NAME = "ml_training_dev"
SOURCE_SCHEMA_NAME = "silver"
TARGET_SCHEMA_NAME = "gold"
TABLE_TARGET_NAME = "customers_orders_category"

##Create Catalog and schema

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS {}".format(CATALOG_NAME))
spark.sql("CREATE SCHEMA IF NOT EXISTS {}".format(TARGET_SCHEMA_NAME))

##Import Libraries

In [0]:
from pyspark.sql.functions import current_date

##Reading Tables

In [0]:
# silver_df = spark.read.table(f"{catalog}.{source_schema}.{table_source_name}")
df_customers = spark.table(f"{CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.olist_customers").dropDuplicates(["customer_id"]).drop("loadDate")
df_orders = spark.read.table(f"{CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.olist_orders").dropDuplicates(["customer_id", "order_id"]).drop("loadDate")
df_order_items = spark.read.table(f"{CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.olist_order_items").dropDuplicates(["order_id", "product_id"]).drop("loadDate")
df_products= spark.read.table(f"{CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.olist_products").dropDuplicates(["product_id"]).drop("loadDate")
df_order_reviews= spark.read.table(f"{CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.olist_order_reviews").select("order_id", "review_score").dropDuplicates(["order_id"]).dropna()

##Transformations

In [0]:
df_customers_orders = df_customers.join(df_orders,on = ["customer_id"],how = "left")

valida_explosao = records_df(df_customers, df_orders, df_customers_orders)
if valida_explosao:
    print("ERRO: explosão de dados")
else:
    print("OK: explosão de dados")

In [0]:
df_customers_orders_category = (df_customers_orders
                                .join(df_order_items, on = ["order_id"], how = "left")
                                .join(df_products, on = ["product_id"], how = "left")
                                )
valida_explosao = records_df(df_products, df_order_items, df_customers_orders_category)
if valida_explosao:
    print("ERRO: explosão de dados")
else:
    print("OK: explosão de dados")

In [0]:
df_final = (df_customers_orders_category
            .join(df_order_reviews, on = ["order_id"], how = "left")
            .dropna()
            .withColumn("loadDate", current_date())
            .withColumn("review_score", df_order_reviews["review_score"].cast("int"))
            )

##Writting tables

In [0]:
df_final.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TABLE_TARGET_NAME}")
print("table successfully created!")

In [0]:
# df_final.display()

In [0]:
# from pyspark.sql.functions import to_date, countDistinct, col, month, year

# df = (spark.table(f"{CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TABLE_TARGET_NAME}")
#       .select("customer_unique_id", 
#             "product_id",
#             "order_purchase_timestamp")
#     .withColumn("order_purchase_m", month(to_date("order_purchase_timestamp")))
#     .withColumn("order_purchase_y", year(to_date("order_purchase_timestamp")))
#  ).groupBy("order_purchase_m", "order_purchase_y").agg(countDistinct("product_id", "customer_unique_id").alias("qtd"))

# df.orderBy(col("order_purchase_m").desc()).display()